# SoccerTwos: Causal GPT-RL vs stock ML-Agents policy

This tutorial evaluates a shared decentralized Causal policy against the stock SoccerTwos ONNX policy. Each Unity scene has eight independent 2-vs-2 fields. Causal controls one complete team (16 agents) and stock controls the opposing 16 agents, then the evaluator swaps sides.

> The published Causal model is the completed training-run checkpoint. One context-32, BOS-discard side-swapped smoke run produced 4 wins and 12 losses against the stock release-23 policy (25.00% win rate). The original safetensors bundle produced the same W/D/L through the cached PolicyRunner. `--seed 0` controlled stock-policy action sampling; the Unity wrapper used deterministic launch seeds 100 and 101, so this is not a multi-environment-seed benchmark.

## Decentralized data and batched inference

Collection stores one ego-centric episode per agent. `manifest.jsonl` links the four trajectories using `match_id`, `field_id`, `team_id`, and `group_id`; the Minari training input contains the independent agent episodes. A single shared model is trained over those episodes. At inference its 16 rows are eight fields times two controlled teammates, each with an independent 32-step context. Rows do not cross-attend.

In [ ]:
%pip install -q -r examples/unity_collection/requirements-collect.txt huggingface_hub

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

root = Path("hf_unity")
causal = Path(hf_hub_download(
    repo_id="ccnets/causal-gpt-rl-unity",
    filename="soccer-twos/soccertwos-b16.onnx",
    local_dir=root / "model",
))
snapshot_download(
    repo_id="ccnets/causal-gpt-rl-unity-envs",
    repo_type="dataset",
    allow_patterns=["SoccerTwos/**", "SoccerTwos.onnx"],
    local_dir=root / "envs",
)
build = root / "envs/SoccerTwos/UnityEnvironment.exe"
stock = root / "envs/SoccerTwos.onnx"
build, causal, stock

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(str(causal), providers=["CPUExecutionProvider"])
[(item.name, item.shape) for item in session.get_inputs()], [(item.name, item.shape) for item in session.get_outputs()]

The expected Causal contract is `states [16,32,336]`, `actions [16,32,9]`, `is_bos [16,32,1]`, `mask [16,32]`, and output `action [16,9]`. The nine output values are three logits for each branch of `MultiDiscrete([3,3,3])`.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "examples/unity/evaluate_matchup.py",
    "--build", str(build),
    "--causal-onnx", str(causal),
    "--stock-onnx", str(stock),
    "--causal-team", "both",
    "--bos-cache-mode", "discard",
    "--stock-baseline",
], check=True)

## Interpreting the result

Use the side-swapped aggregate W/D/L as the primary comparison. The stock-vs-stock run is a symmetry and evaluator sanity check, not a learned-policy score. Run more seeds or Unity instances before reporting a statistically stable result; the eight fields in one launch are only a smoke test.